# Qwen3 Transformers 部署与交互教程

本教程演示如何使用 **HuggingFace Transformers** 框架部署 Qwen3 大模型，包括：

1. 环境准备与依赖安装
2. 模型下载
3. 模型加载与文本生成
4. 流式输出

> **适用场景**：Mac CPU 推理（通用兼容），也可在 Linux GPU 环境使用  
> **推荐模型**：`Qwen3-8B`  
> **注意**：Mac 上 Transformers 默认使用 CPU 推理，速度较 MLX 慢，建议优先使用 MLX 版本

## 1. 环境准备

安装所需依赖：

In [1]:
!pip install torch transformers psutil -q

## 2. 模型下载

使用 `modelscope` 从国内源下载模型到本地 `models/` 目录：

In [2]:
!pip install modelscope -q

In [3]:
from modelscope import snapshot_download
import time

MODEL_NAME = "Qwen3-8B"
LOCAL_DIR = "../models"
REPO_ID = f"Qwen/{MODEL_NAME}"

MODEL_PATH = f"{LOCAL_DIR}/{MODEL_NAME}"

print(f"📋 Repo ID:   {REPO_ID}")
print(f"📋 本地路径:  {MODEL_PATH}")
print()

s = time.time()
# snapshot_download(repo_id=REPO_ID, local_dir=MODEL_PATH)
print(f"\n✅ 下载完成，耗时 {time.time()-s:.2f} 秒")

📋 Repo ID:   Qwen/Qwen3-8B
📋 本地路径:  ../models/Qwen3-8B


✅ 下载完成，耗时 0.00 秒


## 3. 加载模型

使用 Transformers 加载模型和 tokenizer。Mac 上默认使用 CPU + float32 推理：

In [ ]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cpu"
dtype = torch.float32

print(f"🚀 开始加载模型 {MODEL_NAME} ...")
start = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH  ,
    dtype=dtype,
    device_map=device,
    trust_remote_code=True
)
model.eval()

print(f"✅ 模型加载完成，耗时 {time.time() - start:.2f} 秒")

🚀 开始加载模型 Qwen3-8B ...


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

✅ 模型加载完成，耗时 10.99 秒


## 4. 文本生成

构建对话 prompt 并调用模型生成回复：

In [ ]:
import psutil
import os

question = "请用一句话解释什么是人工智能？"

# 构建对话 prompt
messages = [
    {"role": "system", "content": "你是一个智能助手。"},
    {"role": "user", "content": question}
]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

# 统计 prompt tokens
inputs = tokenizer(prompt, return_tensors="pt")
prompt_tokens = inputs["input_ids"].shape[1]

# 生成
start = time.time()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        do_sample=True,
        eos_token_id=[151645, 151643],
        pad_token_id=tokenizer.pad_token_id
    )
gen_time = time.time() - start

# 解码输出（只取新生成的部分）
generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)
response_tokens = len(generated_ids)

# 打印结果
print("🤖 模型回复")
print("=" * 50)
print(response)
print("=" * 50)

# 性能统计
memory_gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3

print(f"\n📊 性能统计")
print("-" * 30)
print(f"  模型:            {MODEL_NAME}")
print(f"  设备:            {device}")
print(f"  Prompt tokens:   {prompt_tokens}")
print(f"  生成 tokens:     {response_tokens}")
print(f"  生成总耗时:      {gen_time:.2f}s")
print(f"  生成速度:        {response_tokens / gen_time:.1f} tokens/s")
print(f"  进程内存:        {memory_gb:.2f} GB")
print("-" * 30)

🤖 模型回复
人工智能是模拟人类智能的计算机系统，能够执行诸如学习、推理、问题解决和感知等任务。

📊 性能统计
------------------------------
  模型:            Qwen3-8B
  设备:            cpu
  Prompt tokens:   29
  生成 tokens:     24
  生成总耗时:      15.10s
  生成速度:        1.6 tokens/s
  进程内存:        28.44 GB
------------------------------
